# Task 14 — External NTP vs NCP audit (canonical)

This notebook runs the smallest experiment capable of answering the central question without
reusing training targets for evaluation.

**Primary hypothesis.** On held-out SWORDS targets, gold-inclusive exact sequence marginalization
(`M-I`) improves acceptable-alternative probability and official ranking GAP over matched
gold-continuation NTP (`C0`) and gold-exclusive marginalization (`M-X`), while preserving the
observed target.

Benchmark roles:

- **SWORDS:** primary human-labelled contextual substitution benchmark.
- **HyperLex:** directional hypernym evaluation; not synthetic NCP training data.
- **bm-semlex:** 200-row wrong-sense guardrail, not a headline benchmark.

Arms, all trained on the same matched volume (`BUDGET` = 1,607 rows, section 2b):

| Arm | Owner | Training signal |
|---|---|---|
| B0 | base | untouched base model |
| A0 | paper | NTP on repeated originals |
| A1 | paper | NCP data augmentation, one sentence per alternative |
| P1 | paper | written mean-log NCP loss, no CLM term |
| C0 | ours | matched gold-continuation NTP |
| M-X / M-I | ours | set marginal, gold-exclusive / gold-inclusive |
| M-P | ours | per-word (paper_mean) **plus** CLM retention, gold-inclusive |
| M-C | ours | set marginal + InfoNCE against SWORDS' human-rejected substitutes |
| M-PC | ours | per-word + retention **+** InfoNCE — completes the 2x2 |

`M_P` vs `M_I` is the loss-form-only comparison Task 13 lacked: same rows, same replay, same base
weight, same candidate set, only the concept loss form differs. Task 13 found the set marginal's
gradient is a softmax over candidates, so it concentrates on the already-preferred member; the
per-word form weights every candidate equally. `M_C` is the project's first contrastive arm
trained on **human** negatives rather than WordNet-mined ones.

The four ours-arms form a 2x2 over objective x contrastive, so the best combination is measured
rather than inferred:

|  | no contrastive | + InfoNCE |
|---|---|---|
| set marginal | `M_I` | `M_C` |
| per-word + retention | `M_P` | `M_PC` |

`C0` (matched gold-continuation NTP) and `M_X` (gold-exclusive marginal) remain the controls, and
`CORE_ARMS` pins the predeclared gate in section 7 to `C0`/`M_X`/`M_I` so the added arms cannot
move the pass/fail line after the fact. Every ours-arm is compared against every paper arm, and
those comparisons are declared here, before any result is read.

**New SWORDS metric.** `rejected_mass_share` is the fraction of acceptable-plus-rejected
probability sitting on substitutes human annotators marked unacceptable (lower is better, the
same direction as the NLLs). `auroc` already showed whether rejected substitutes *rank* below
acceptable ones; it cannot show whether their probability was actually suppressed, which is the
specific claim an InfoNCE arm makes. The two come apart: a model can rank perfectly
(`auroc = 1.0`) while holding most of the mass on rejected candidates.

**HyperLex is reported, not gated.** These arms train on SWORDS synonymy, so HyperLex measures
*off-relation transfer* and a flat result is the expected outcome — a retention check that
concept training did not disturb hypernym directionality. Phase C is where HyperLex becomes the
matched benchmark, against arms trained on `hyp/youtube_clean_gold`.

SWORDS concept sets are human acceptability judgements, so the paper's Appendix-A LLM extraction
step does not apply here; `A1`'s corpus is a mechanical substitution over those sets.

The default path evaluates untouched 1B/3B bases, then trains the six 1B SWORDS arms with seed 42.
Official SWORDS test, extra seeds, hypernym transfer, and 3B training are gated off.

Storage invariant: models/checkpoints/caches live only under ephemeral `/content`. Drive receives
JSON/CSV reports only. Training writes one final weight copy, evaluates it, and deletes it in a
`finally` block.

## 0. Runtime, authentication, and repository

In [ ]:
import os, sys, subprocess, shutil, json, glob, hashlib
from pathlib import Path

import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
!pip install -q "transformers>=4.57,<5" datasets accelerate pytest pandas scipy bitsandbytes huggingface_hub

from google.colab import drive, userdata
from huggingface_hub import login
drive.mount('/content/drive')
hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Add HF_TOKEN to Colab Secrets; Llama-3.2 is gated.'
login(token=hf_token, add_to_git_credential=False)


NVIDIA L4, 23034 MiB, 22561 MiB

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
REPO_URL = 'https://github.com/SharvaGogawale1/concept-aware-training.git'
REPO_DIR = Path('/content/concept_aware_training')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

SCRIPTS = REPO_DIR / 'transformers/examples/pytorch/language-modeling'
required = [
    'sequence_ncp_trainer.py', 'run_clm_sequence_ncp.py', 'eval_concept_ppl_v3.py',
    'eval_swords.py', 'eval_hyperlex.py', 'eval_bm_semlex.py',
    'compare_task14_results.py', 'test_sequence_ncp.py', 'test_task14_external.py',
]
missing = [name for name in required if not (SCRIPTS / name).exists()]
for name in ['prepare_swords_concept_sets.py', 'verify_task14_data.py',
             'rebuild_gold_inclusive.py']:
    if not (REPO_DIR / name).exists(): missing.append(name)
assert not missing, f'Push Task-14 files to GitHub before Colab: {missing}'
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)


Repository: /content/concept_aware_training


## 1. Locked configuration

In [ ]:
DATA = REPO_DIR / 'data'
SCRATCH = Path('/content/task14_scratch')
DRIVE_ROOT = Path('/content/drive').resolve()
RESULTS = DRIVE_ROOT / 'MyDrive/concept_aware_outputs/task14_controlled'
SCRATCH.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)

PRIMARY_SEED = 42
# Three seeds: required before the Phase-C SWORDS-test gate can open, and required for M_C to be
# reportable at all.  7 trained arms x 3 seeds ~ 8-9 h -- split across sessions, the resume logic
# skips finished runs.  Set False for a one-seed screen.
RUN_EXTRA_SEEDS = True
SEEDS = [PRIMARY_SEED] + ([7, 123] if RUN_EXTRA_SEEDS else [])
# Skip any run whose result JSONs are all already on Drive.  Cell 5's original comment
# claimed this existed; it did not -- train_one retrained unconditionally, so re-running
# the notebook to pick up a read-side fix cost a full 8-9 h retrain.  Set False to force.
RESUME_FINISHED_RUNS = True
RUN_PHASE_A = True
RUN_GPU_SMOKE = True
RUN_SWORDS_PILOT = True
RUN_PAPER_ARMS = True              # Iyer et al. A0/A1/P1 on the external benchmarks
RUN_SCHEDULE_SWEEP  = False    # section 6a: 4 A0 runs, one seed, ~1.5 h
RUN_ALPHA_SWEEP     = False    # section 6b: M_P at alpha in {0.5, 2, 4}, all seeds, ~3 h
RUN_PHASE_C_SWORDS_TEST = False    # enable only after all three 1B seeds pass
RUN_PHASE_C_HYPERNYM = False       # enable only after the SWORDS gate passes
TRAIN_3B_AFTER_PASS = False        # requires A100 and a replicated 1B result

MODEL_REPOS = {'1B': 'meta-llama/Llama-3.2-1B', '3B': 'meta-llama/Llama-3.2-3B'}
MODEL_PATHS = {key: Path(f'/content/Llama-3.2-{key}') for key in MODEL_REPOS}

def assert_ephemeral(path):
    resolved = Path(path).resolve()
    assert os.path.commonpath([str(resolved), str(DRIVE_ROOT)]) != str(DRIVE_ROOT), \
        f'weights/checkpoints may not be written to Drive: {resolved}'

print('seeds:', SEEDS, '| results:', RESULTS)


seeds: [42, 7, 123] | results: /content/drive/MyDrive/concept_aware_outputs/task14_controlled


## 2. Fetch, verify, split, and test data

In [ ]:
# Pinned URLs and SHA-256 checks; downloads only when a file is absent.
subprocess.run([
    sys.executable, 'verify_task14_data.py', '--download_missing',
    '--report_json', str(RESULTS / 'data_integrity.json')
], check=True)

SWORDS_DEV = DATA / 'swords/swords-v1.1_dev.json.gz'
SWORDS_TEST = DATA / 'swords/swords-v1.1_test.json.gz'
SW_DIR = DATA / 'swords/concept_dev'
subprocess.run([
    sys.executable, 'prepare_swords_concept_sets.py',
    '--swords_json', str(SWORDS_DEV), '--test_json', str(SWORDS_TEST),
    '--out_dir', str(SW_DIR), '--split_seed', '42', '--train_fraction', '0.8'
], check=True)

# Rebuild both synonym and hypernym gold-inclusive views with the boundary-safe recovery code.
subprocess.run([sys.executable, 'rebuild_gold_inclusive.py'], check=True)
report = json.load(open(SW_DIR / 'prepare_report.json'))
assert report['slots_train'] == 251 and report['slots_val'] == 62
assert report['augmentation_lines']['train'] == 1607
assert report['dev_test_disjointness'] == {
    'target_id_overlap_with_dev': 0, 'context_id_overlap_with_dev': 0}
print(json.dumps(report, indent=2))


{
  "source": "/content/concept_aware_training/data/swords/swords-v1.1_dev.json.gz",
  "source_sha256": "bc3570786455a05f11fcb4fc7f54e117642d5fb5e98f19355028c5c7ee00c954",
  "test_source": "/content/concept_aware_training/data/swords/swords-v1.1_test.json.gz",
  "test_source_sha256": "f94b6b73c52fd4bd549180bc119d44e9f9097ec78240a8b42363add158e58480",
  "positive_threshold": 0.5,
  "negative_threshold": 0.0,
  "min_positives": 2,
  "max_negatives": 10,
  "pos_filter": null,
  "split_seed": 42,
  "train_fraction": 0.8,
  "slots_kept": 313,
  "slots_train": 251,
  "slots_val": 62,
  "augmentation_lines": {
    "train": 1607,
    "val": 464,
    "full": 2071
  },
  "dev_test_disjointness": {
    "target_id_overlap_with_dev": 0,
    "context_id_overlap_with_dev": 0
  },
  "mean_positives": 6.617,
  "mean_negatives": 9.326,
  "median_positives": 6,
  "pos_distribution": {
    "NOUN": 112,
    "ADJ": 48,
    "VERB": 125,
    "ADV": 28
  },
  "counts": {
    "candidate_dropped_duplicate": 5,
 

### 2b. Matched training volume for the Iyer et al. arms

Iyer et al. hold training-instance volume approximately fixed across variants (paper section 2.2),
and the augmentation corpus is what fixes it: A1 is defined as one sentence per alternative, so its
size is not a free parameter. Here that is **1,607** rows.

Every other arm is therefore padded to 1,607 with deterministically repeated originals. Without
this, A1 would train on 3.7x the rows of `C0`/`M_X`/`M_I` and any A1 advantage would be
indistinguishable from seeing more data — the same confound Task 6 was written to rule out.

This matches example count, not token count, which is the paper's own protocol.

In [ ]:
import pandas as pd, numpy as np

def read_lines(path):
    return [line.strip() for line in open(path, encoding='utf-8') if line.strip()]

def write_lines(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n'.join(rows) + '\n', encoding='utf-8')

def cycle_to_size(rows, size):
    assert rows and size >= 0
    return [rows[index % len(rows)] for index in range(size)]

SW_AUG_TRAIN = SW_DIR / 'context_syn_train.txt'
BUDGET = len(read_lines(SW_AUG_TRAIN))
assert BUDGET == report['augmentation_lines']['train']

originals = read_lines(SW_DIR / 'vanilla_train.txt')
concept_rows = pd.read_csv(SW_DIR / 'context_loss_train_goldexcl.csv')
N_CONCEPT = len(concept_rows)
assert N_CONCEPT < BUDGET, 'replay padding would be negative'

# A0: repeated originals only.  Matched replay: what the hybrid arms add on top of their concept
# rows to reach the same budget.  P1: the paper's gold-exclusive concept rows, cycled to budget.
A0_TRAIN = SCRATCH / 'swords_A0_repeated_original.txt'
MATCHED_REPLAY = SCRATCH / 'swords_matched_replay.txt'
P1_TRAIN = SCRATCH / 'swords_P1_repeated_concepts.csv'
write_lines(A0_TRAIN, cycle_to_size(originals, BUDGET))
write_lines(MATCHED_REPLAY, cycle_to_size(originals, BUDGET - N_CONCEPT))

repeated = concept_rows.iloc[np.arange(BUDGET) % N_CONCEPT].copy()
repeated['row_id'] = [f'{row_id}:paper-repeat:{index:05d}'
                      for index, row_id in enumerate(repeated.row_id.astype(str))]
repeated.to_csv(P1_TRAIN, index=False)

print(f'budget N={BUDGET} | concept rows={N_CONCEPT} | replay={BUDGET - N_CONCEPT}')


budget N=1607 | concept rows=251 | replay=1356


In [ ]:
# CPU regression tests: exact scores, gold-slot labels, caching, gradients, GAP fixtures,
# split disjointness, external file hashes, and candidate microbatch equivalence.
subprocess.run([
    sys.executable, '-m', 'pytest', '-q', '--confcutdir=.',
    str(SCRIPTS / 'test_sequence_ncp.py'), str(SCRIPTS / 'test_task14_external.py')
], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', '--confcutdir=.', '/content/concept_aware_training/transformers/examples/pytorch/language-modeling/test_sequence_ncp.py', '/content/concept_aware_training/transformers/examples/pytorch/language-modeling/test_task14_external.py'], returncode=0)

## 3. Download untouched base models to ephemeral storage

In [ ]:
from huggingface_hub import snapshot_download
for key in ['1B', '3B']:
    snapshot_download(
        repo_id=MODEL_REPOS[key], local_dir=str(MODEL_PATHS[key]), token=hf_token,
        ignore_patterns=['*.msgpack', '*.h5', '*.ot', 'original/*'],
    )
    print(key, MODEL_PATHS[key])


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

LICENSE.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

3B /content/Llama-3.2-3B


## 4. External and intrinsic evaluation functions

In [ ]:
HYPERLEX_LEX = DATA / 'hyperlex-data/splits/lexical/hyperlex_test_all_lexical.txt'
HYPERLEX_RANDOM = DATA / 'hyperlex-data/splits/random/hyperlex_test_all_random.txt'
BM_TSV = DATA / 'bm_semlex/curated_200.tsv'

def run_command(parts):
    print(' '.join(map(str, parts)))
    subprocess.run([str(part) for part in parts], check=True)

def run_artifacts(tag, evaluation='dev', trained=True):
    """Every JSON a completed run leaves in RESULTS, so resume can tell finished from partial.

    Test-mode runs deliberately produce fewer files: external_eval is called with
    include_hyperlex=False and intrinsic_eval is not called at all, so requiring those here
    would make a finished test run look unfinished and retrain it forever.
    """
    names = ['swords', 'bm_semlex']
    if evaluation != 'test':
        names += ['hyperlex_lexical', 'hyperlex_random', 'intrinsic']
    paths = [RESULTS / f'{name}__{tag}.json' for name in names]
    if trained:
        paths.append(RESULTS / f'train__{tag}.json')
    return paths

def already_done(tag, evaluation='dev', trained=True):
    # All-or-nothing on purpose: train_one deletes the checkpoint when it finishes, so a run
    # missing even one eval JSON cannot be re-evaluated without retraining.
    return RESUME_FINISHED_RUNS and all(path.exists()
                                        for path in run_artifacts(tag, evaluation, trained))

def external_eval(tag, checkpoint, model_key, swords_json, target_ids_file=None,
                  full_swords=False, include_hyperlex=True):
    tokenizer = MODEL_PATHS[model_key]
    batch = 32 if model_key == '1B' else 16
    swords_out = RESULTS / f'swords__{tag}.json'
    command = [sys.executable, SCRIPTS / 'eval_swords.py', '--checkpoints', checkpoint,
               '--tokenizer_path', tokenizer, '--swords_json', swords_json,
               '--modes', 'left']
    if full_swords: command += ['full']
    if target_ids_file: command += ['--target_ids_file', target_ids_file]
    command += ['--block_size', '256', '--batch_size', batch,
                '--n_bootstrap', '2000', '--results_json', swords_out]
    run_command(command)

    if include_hyperlex:
        for split, path in [('lexical', HYPERLEX_LEX), ('random', HYPERLEX_RANDOM)]:
            run_command([sys.executable, SCRIPTS / 'eval_hyperlex.py', '--checkpoints', checkpoint,
                         '--tokenizer_path', tokenizer, '--hyperlex', path, '--pos', 'N',
                         'V',
                         '--batch_size', batch, '--n_bootstrap', '2000',
                         '--results_json', RESULTS / f'hyperlex_{split}__{tag}.json'])

    run_command([sys.executable, SCRIPTS / 'eval_bm_semlex.py', '--checkpoints', checkpoint,
                 '--tokenizer_path', tokenizer, '--data', BM_TSV, '--modes', 'left',
                 '--batch_size', batch, '--results_json', RESULTS / f'bm_semlex__{tag}.json'])

def intrinsic_eval(tag, checkpoint, model_key, concept_csv, relation='swords'):
    run_command([sys.executable, SCRIPTS / 'eval_concept_ppl_v3.py',
                 '--checkpoints', checkpoint, '--tokenizer_path', MODEL_PATHS[model_key],
                 '--concept_csv', f'{relation}={concept_csv}',
                 '--vanilla_val', DATA / 'syn/youtube_clean_gold/vanilla_val.txt',
                 '--batch_size', '16', '--n_bootstrap', '2000',
                 '--results_json', RESULTS / f'intrinsic__{tag}.json'])


## 5. Phase A — untouched 1B/3B evaluation validation

In [ ]:
if RUN_PHASE_A:
    for model_key in ['1B', '3B']:
        tag = f'base_{model_key}'
        if already_done(tag, trained=False):
            print('resume: already complete, skipping', tag); continue
        external_eval(tag, MODEL_PATHS[model_key], model_key, SWORDS_DEV,
                      SW_DIR / 'target_ids_val.txt')
        intrinsic_eval(tag, MODEL_PATHS[model_key], model_key,
                       SW_DIR / 'context_loss_val.csv')
print('Phase A complete. Scale differences are descriptive, not a metric-validity gate.')


/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/eval_swords.py --checkpoints /content/Llama-3.2-1B --tokenizer_path /content/Llama-3.2-1B --swords_json /content/concept_aware_training/data/swords/swords-v1.1_dev.json.gz --modes left --target_ids_file /content/concept_aware_training/data/swords/concept_dev/target_ids_val.txt --block_size 256 --batch_size 32 --n_bootstrap 2000 --results_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/swords__base_1B.json
/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/eval_hyperlex.py --checkpoints /content/Llama-3.2-1B --tokenizer_path /content/Llama-3.2-1B --hyperlex /content/concept_aware_training/data/hyperlex-data/splits/lexical/hyperlex_test_all_lexical.txt --pos N V --batch_size 32 --n_bootstrap 2000 --results_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/hyperlex_lexical__base_1B.json
/usr/bin/python3 /conten

## 6. Phase B — 1B C0/M-X/M-I pilot

In [ ]:
# `data` selects which matched view an arm trains on:
#   hybrid       concept rows + repeated-original replay, padded to BUDGET  (ours)
#   vanilla      repeated originals only                                    (paper NTP baseline)
#   augmentation one sentence per alternative                               (paper NCP data aug.)
#   paper        gold-exclusive concept rows cycled to BUDGET               (paper NCP loss)
#
# base_weight is passed explicitly for every arm.  P1 is the paper's written equation, which
# carries no CLM term at all; leaving base_weight at its 1.0 default would silently train this
# project's hybrid CLM+NCP objective and label it as the paper reconstruction.
ARMS = {
    'C0':  {'objective': 'none', 'alpha': 0.0, 'base_weight': 1.0,
            'view': 'inclusive', 'data': 'hybrid', 'owner': 'ours'},
    'M_X': {'objective': 'set_marginal', 'alpha': 0.5, 'base_weight': 1.0,
            'view': 'exclusive', 'data': 'hybrid', 'owner': 'ours'},
    'M_I': {'objective': 'set_marginal', 'alpha': 0.5, 'base_weight': 1.0,
            'view': 'inclusive', 'data': 'hybrid', 'owner': 'ours'},
    'A0':  {'objective': 'none', 'alpha': 0.0, 'base_weight': 1.0,
            'view': 'inclusive', 'data': 'vanilla', 'owner': 'paper'},
    'A1':  {'objective': 'none', 'alpha': 0.0, 'base_weight': 1.0,
            'view': 'inclusive', 'data': 'augmentation', 'owner': 'paper'},
    'P1':  {'objective': 'paper_mean', 'alpha': 1.0, 'base_weight': 0.0,
            'view': 'exclusive', 'data': 'paper', 'owner': 'paper'},
    # M_P: the paper's per-word loss WITH a CLM retention term, on the same rows and replay as
    # M_I.  Task 13 showed the set marginal's gradient is a softmax over candidates, so it
    # concentrates on the already-preferred member instead of spreading mass across the set;
    # the per-word form weights every candidate equally.  P1 collapses NTP only because it
    # carries no CLM term (base_weight 0).  M_P separates the objective from that collapse, and
    # M_P vs M_I is the loss-form-only comparison at matched data.
    'M_P': {'objective': 'paper_mean', 'alpha': 1.0, 'base_weight': 1.0,
            'view': 'inclusive', 'data': 'hybrid', 'owner': 'ours'},
    # M_C: InfoNCE against SWORDS' HUMAN-REJECTED substitutes.  This is the first contrastive arm
    # in the project trained on real negatives: prepare_swords_concept_sets keeps substitutes
    # scored at or below the negative threshold by the SWORDS annotators (2,919 of them), rather
    # than WordNet co-hyponyms, whose miner the July 2026 audit found structurally broken.
    'M_C': {'objective': 'set_marginal', 'alpha': 0.5, 'base_weight': 1.0, 'contrast_beta': 1.0,
            'view': 'inclusive', 'data': 'contrastive', 'owner': 'ours'},
    # M_PC completes the 2x2 over {set marginal, per-word} x {no contrastive, contrastive}.
    # Without it the best-performing combination can only be inferred: Task 13 says per-word is
    # the better base objective and M_C tests contrastive on the set marginal, but nothing tests
    # the two together, which is the configuration the project actually proposes.
    'M_PC': {'objective': 'paper_mean', 'alpha': 1.0, 'base_weight': 1.0, 'contrast_beta': 1.0,
             'view': 'inclusive', 'data': 'contrastive', 'owner': 'ours'},
}
PAPER_ARMS = [arm for arm, spec in ARMS.items() if spec['owner'] == 'paper']
# CORE_ARMS are the three the predeclared gate in section 7 is written against; adding arms
# below must not be able to move that pass/fail line after the fact.  Phase C expands CORE only.
CORE_ARMS = ['C0', 'M_X', 'M_I']
OUR_ARMS = CORE_ARMS + ['M_P', 'M_C', 'M_PC']

# M_P's alpha=1.0 is inherited from the paper's written equation, not tuned -- no coefficient
# search has ever been run on the per-word objective.  These are points on an alpha sweep, not
# separate methods (Task 13b's reporting note).  Deliberately NOT in OUR_ARMS: adding them
# there would pull them into section 7's gate and section 7b's paper contrasts, turning dev
# tuning into headline comparisons.  Pick alpha here, then promote the winner to a real arm.
ALPHA_GRID = [0.5, 2.0, 4.0]
ALPHA_ARMS = []
for _alpha in ALPHA_GRID:
    _key = f"M_P_a{str(_alpha).replace('.', 'p')}"
    ARMS[_key] = {**ARMS['M_P'], 'alpha': _alpha, 'owner': 'sweep'}
    ALPHA_ARMS.append(_key)

# Intrinsic evaluation is held fixed per source so every arm is scored on identical rows.
INTRINSIC_VAL = {
    'swords': SW_DIR / 'context_loss_val.csv',
    'swords_full': SW_DIR / 'context_loss_val.csv',
    'hypernym': DATA / 'hyp/youtube_clean_gold/context_loss_val.csv',
}

def arm_files(arm, source='swords'):
    spec = ARMS[arm]
    suffix = '_goldexcl' if spec['view'] == 'exclusive' else ''
    # MUST precede the `data != 'hybrid'` branch below: 'contrastive' is not 'hybrid', so falling
    # through would KeyError on the paper-view lookup table.
    if spec['data'] == 'contrastive':
        if source != 'swords':
            raise ValueError(f'contrastive arm {arm} has no matched view for source={source}')
        # Same replay, and therefore the same matched budget, as the hybrid arms; only the
        # candidate/negative columns differ.
        return {'train': SW_DIR / 'contrastive_train.csv',
                'val': SW_DIR / 'contrastive_val.csv', 'replay': MATCHED_REPLAY}
    if spec['data'] != 'hybrid':
        # The budget-matched paper views are built for the SWORDS pilot only. Phase C sources
        # would need their own matched corpora; fail loudly rather than train on a mismatch.
        if source != 'swords':
            raise ValueError(f'paper arm {arm} has no matched view for source={source}')
        train = {'vanilla': A0_TRAIN, 'augmentation': SW_AUG_TRAIN, 'paper': P1_TRAIN}[spec['data']]
        val = (SW_DIR / f'context_loss_val{suffix}.csv' if spec['data'] == 'paper'
               else SW_DIR / 'vanilla_val.txt')
        return {'train': train, 'val': val, 'replay': None}
    if source == 'swords':
        root, train_name, replay = SW_DIR, 'train', MATCHED_REPLAY
    elif source == 'swords_full':
        root, train_name, replay = SW_DIR, 'full', SW_DIR / 'vanilla_full.txt'
    elif source == 'hypernym':
        root = DATA / 'hyp/youtube_clean_gold'
        train_name, replay = 'train', root / 'vanilla_train.txt'
    else:
        raise ValueError(source)
    return {
        'train': root / f'context_loss_{train_name}{suffix}.csv',
        'val': root / f'context_loss_val{suffix}.csv',
        'replay': replay,
    }

def arms_for_source(arms, source):
    """Drop arms that have no budget-matched view for `source`, and say which.

    `arm_files` raises for an arm/source pair with no matched corpus -- that assert is the
    tripwire that stops a mismatched training run, and it stays.  But the Phase-C loops iterate
    OUR_ARMS wholesale, and the contrastive arms (M_C, M_PC) are built only on SWORDS, whose
    negatives are human-annotated.  Without this filter, enabling RUN_PHASE_C_HYPERNYM aborts
    the whole block on the first contrastive arm instead of running the arms that do have a
    hypernym view.  Building a hypernym contrastive view is a data task, not a loop fix: it
    needs mined negatives for hyp/youtube_clean_gold, and the July 2026 audit found that miner
    broken.  Until that exists, skip loudly rather than crash or silently train on a mismatch.
    """
    usable, skipped = [], []
    for arm in arms:
        try:
            arm_files(arm, source)
        except ValueError as error:
            skipped.append((arm, str(error)))
        else:
            usable.append(arm)
    for arm, reason in skipped:
        print(f'skipping {arm} for source={source}: {reason}')
    return usable

SCHEDULE = [
    '--num_train_epochs', '2', '--learning_rate', '1e-5', '--warmup_ratio', '0.1',
    '--block_size', '128', '--per_device_train_batch_size', '1',
    '--per_device_eval_batch_size', '1', '--gradient_accumulation_steps', '16',
    '--logging_steps', '25', '--save_strategy', 'no', '--save_only_model',
    '--bf16', '--overwrite_output_dir',
    '--do_train', '--do_eval', '--report_to', 'none',
    '--forbidden_output_root', str(DRIVE_ROOT),
    # Required, not cosmetic: text-row dedup defaults ON, and would collapse A0's 1,607 repeated
    # originals and the matched replay back to 251 unique lines, silently undoing the volume
    # matching with no error raised. The rows_raw assertion in train_one is the tripwire.
    '--no-deduplicate_text_rows',
]

def schedule_with(epochs=None, lr=None):
    """SCHEDULE with the two swept values substituted; everything else stays locked."""
    parts = list(SCHEDULE)
    for flag, value in (('--num_train_epochs', epochs), ('--learning_rate', lr)):
        if value is not None:
            parts[parts.index(flag) + 1] = str(value)
    return parts

def train_one(arm, seed, source='swords', smoke=False, model_key='1B', evaluation='dev',
              epochs=None, lr=None):
    spec, files = ARMS[arm], arm_files(arm, source)
    # A swept schedule gets its own tag suffix.  Without this, a sweep run would overwrite the
    # locked-schedule run of the same arm/seed and silently corrupt the 27 results already on
    # Drive; with it, the default path keeps its existing tags and stays resumable.
    assert (epochs is None) == (lr is None), 'sweep both epochs and lr, or neither'
    sched = '' if epochs is None else f'_e{epochs:g}_lr{lr:g}'
    tag = f'{model_key}_{source}_{arm}_s{seed}{sched}' + ('_smoke' if smoke else '')
    if not smoke and already_done(tag, evaluation):
        print('resume: already complete, skipping', tag)
        return tag
    output = SCRATCH / tag
    cache = SCRATCH / f'cache_{source}'
    assert_ephemeral(output); assert_ephemeral(cache)
    optimizer = 'adamw_bnb_8bit' if model_key == '3B' else 'adamw_torch'
    # Both settings are numerics-preserving, so they do not affect the comparison:
    # gradient checkpointing only trades recompute for memory, and candidate microbatching is
    # a pure forward-splitting loop (test_candidate_microbatching_matches_one_full_forward).
    # At batch 1 x 128 tokens a 1B model peaks near 11 GiB, so checkpointing buys nothing and
    # costs ~25%. The gated 3B path keeps both conservative settings.
    heavy = model_key == '3B'
    candidate_microbatch = '1' if heavy else '8'
    checkpointing = '--gradient_checkpointing' if heavy else '--no-gradient_checkpointing'
    command = [sys.executable, SCRIPTS / 'run_clm_sequence_ncp.py',
               '--model_name_or_path', MODEL_PATHS[model_key], '--tokenizer_name', MODEL_PATHS[model_key],
               '--train_file', files['train'], '--validation_file', files['val'],
               '--objective', spec['objective'],
               '--ncp_alpha', spec['alpha'], '--base_loss_weight', spec['base_weight'],
               '--required_coverage', '0.99',
               '--contrast_beta', spec.get('contrast_beta', 0.0),
               '--preprocessing_cache_dir', cache, '--optim', optimizer,
               '--candidate_microbatch_size', candidate_microbatch, checkpointing,
               '--seed', seed, '--output_dir', output] + schedule_with(epochs, lr)
    if spec['data'] == 'contrastive':
        # contrastive_train.csv names its columns positives/negatives, not context_syn.
        command += ['--candidate_column', 'positives', '--negative_column', 'negatives']
    if files['replay']:
        command += ['--replay_file', files['replay']]
    if smoke:
        command += ['--max_train_samples', '8', '--max_eval_samples', '8',
                    '--num_train_epochs', '1']
    if evaluation == 'test':
        command += ['--no-do_eval']  # fixed epochs; never select on the official test
    try:
        run_command(command)
        summary = json.load(open(output / 'sequence_ncp_run.json'))
        if spec['objective'] != 'none':
            assert summary['training_concept_coverage']['coverage'] >= 0.99
        if not smoke and source == 'swords':
            rows_raw = summary['train_preprocessing']['rows_raw']
            assert rows_raw == BUDGET, (
                f'{arm} trained on {rows_raw} rows, expected the matched budget {BUDGET}; '
                'check that --no-deduplicate_text_rows is still in SCHEDULE')
        shutil.copy2(output / 'sequence_ncp_run.json', RESULTS / f'train__{tag}.json')
        if not smoke:
            if evaluation == 'test':
                external_eval(tag, output, model_key, SWORDS_TEST, include_hyperlex=False)
            else:
                # HyperLex on every trained arm: Phase A reports base-model numbers, so without
                # this the hypernym benchmark has nothing to be compared against. ~926 pairs.
                external_eval(tag, output, model_key, SWORDS_DEV, SW_DIR / 'target_ids_val.txt',
                              include_hyperlex=True)
                # One fixed concept set per source, never the arm's own validation file:
                # A0/A1 validate on plain text, and M_X on the gold-exclusive view, so passing
                # files['val'] here would both crash the txt arms and score the rest on
                # different data. Phase A uses this same file, so bases stay comparable too.
                relation = 'hypernym' if source == 'hypernym' else 'swords'
                intrinsic_eval(tag, output, model_key, INTRINSIC_VAL[source], relation=relation)
        return tag
    finally:
        shutil.rmtree(output, ignore_errors=True)
        print('deleted ephemeral model:', output)

if RUN_GPU_SMOKE:
    train_one('M_I', PRIMARY_SEED, smoke=True)


/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/run_clm_sequence_ncp.py --model_name_or_path /content/Llama-3.2-1B --tokenizer_name /content/Llama-3.2-1B --train_file /content/concept_aware_training/data/swords/concept_dev/context_loss_train.csv --validation_file /content/concept_aware_training/data/swords/concept_dev/context_loss_val.csv --objective set_marginal --ncp_alpha 0.5 --base_loss_weight 1.0 --required_coverage 0.99 --contrast_beta 0.0 --preprocessing_cache_dir /content/task14_scratch/cache_swords --optim adamw_torch --candidate_microbatch_size 8 --no-gradient_checkpointing --seed 42 --output_dir /content/task14_scratch/1B_swords_M_I_s42_smoke --num_train_epochs 2 --learning_rate 1e-5 --warmup_ratio 0.1 --block_size 128 --per_device_train_batch_size 1 --per_device_eval_batch_size 1 --gradient_accumulation_steps 16 --logging_steps 25 --save_strategy no --save_only_model --bf16 --overwrite_output_dir --do_train --do_eval --report

### 6a. Schedule sweep — the gate Task 13b has and Task 14 never did

Every trained arm in this notebook is worse than the untouched base on general NTP
(2.10-2.28 vs 2.060), while every arm in Task 13b was *better* than the same base on the
same eval file. Task 14 trains 2 epochs at 1e-5 — a point 13b never swept, and whose two
nearest neighbours both failed 13b's gate. The gap is confounded with domain shift (this
notebook trains on SWORDS and measures NTP on YouTube), so it is not proof of
over-training — but nothing here can currently tell the two apart.

Sweep `A0`, the objective-free paper baseline, so the schedule is isolated from the loss.
Gate: NTP NLL at or below the untouched base. One seed, 4 runs.


In [ ]:
SWEEP_GRID = [(1, 1e-5), (1, 5e-6), (2, 5e-6), (3, 3e-6)]

sweep_rows = []
if RUN_SCHEDULE_SWEEP:
    BASE_NTP = json.load(open(RESULTS / 'intrinsic__base_1B.json'))[0]['ntp']['ntp_nll_mean']

    # The locked schedule's A0 run already exists on Drive under its unsuffixed tag; read it as
    # the reference row rather than retraining an identical model under a swept tag.
    current = RESULTS / f'intrinsic__1B_swords_A0_s{PRIMARY_SEED}.json'
    if current.exists():
        sweep_rows.append({'epochs': 2, 'lr': 1e-5, 'note': 'locked SCHEDULE',
                           **json.load(open(current))[0]['ntp']})

    for epochs, lr in SWEEP_GRID:
        print(f'\n=== sweep A0 epochs={epochs} lr={lr:g} ===', flush=True)
        tag = train_one('A0', PRIMARY_SEED, epochs=epochs, lr=lr)
        sweep_rows.append({'epochs': epochs, 'lr': lr, 'note': '',
                           **json.load(open(RESULTS / f'intrinsic__{tag}.json'))[0]['ntp']})

    sweep = pd.DataFrame(sweep_rows).drop_duplicates(subset=['epochs', 'lr'], keep='first')
    sweep['beats_base'] = sweep.ntp_nll_mean <= BASE_NTP
    sweep.insert(0, 'B0 NTP NLL', BASE_NTP)
    sweep.to_csv(RESULTS / 'task14_schedule_sweep.csv', index=False)
    print('\n' + sweep[['epochs', 'lr', 'note', 'ntp_nll_mean', 'ntp_ppl',
                         'ntp_accuracy', 'beats_base']].to_string(index=False))

    passing = sweep[sweep.beats_base]
    if passing.empty:
        best = sweep.loc[sweep.ntp_nll_mean.idxmin()]
        print(f'\nGATE FAILED. Closest: epochs={best.epochs:g} lr={best.lr:g} '
              f'NTP NLL {best.ntp_nll_mean:.4f} vs base {BASE_NTP:.4f} '
              f'(gap {best.ntp_nll_mean - BASE_NTP:+.4f}).')
        print('Widen SWEEP_GRID downward (lr 1e-6/2e-6) before trusting any retention number. '
              'If nothing passes, the gap is domain shift, not the schedule -- say so and move '
              'on; the arms are still comparable to each other at a fixed schedule.')
    else:
        best = passing.loc[passing.ntp_nll_mean.idxmin()]
        print(f'\nGATE PASSED. Best: epochs={best.epochs:g} lr={best.lr:g} '
              f'NTP NLL {best.ntp_nll_mean:.4f} <= base {BASE_NTP:.4f}')
        if (best.epochs, best.lr) != (2, 1e-5):
            print('This is NOT the locked schedule. Every arm must be retrained at it before '
                  'the numbers are comparable: edit SCHEDULE, then set RESUME_FINISHED_RUNS = '
                  'False for one full pass (or write to a fresh RESULTS folder).')
else:
    print('Schedule sweep disabled (RUN_SCHEDULE_SWEEP = False).')


### 6b. Alpha sweep on the per-word objective

`M_P` runs at `alpha=1.0` because that is the coefficient in the paper's equation, not
because anything tested it; `M_X`/`M_I` were given 0.5 by the same inheritance. Task 13b
showed that confusing a coefficient with a method is exactly how `A3` got misread. This is
pre-registered tuning on dev: pick alpha by the summary table, then promote the winner to a
full arm and let sections 7/7b/7c judge it with paired intervals.


In [ ]:
ALPHA_TAGS = []
if RUN_ALPHA_SWEEP:
    for seed in SEEDS:
        for arm in ALPHA_ARMS:
            ALPHA_TAGS.append(train_one(arm, seed))
print('Alpha-sweep arms:', ALPHA_ARMS if RUN_ALPHA_SWEEP else '(disabled)')
print('Alpha-sweep tags:', ALPHA_TAGS)


In [ ]:
PILOT_ARMS = (OUR_ARMS + (PAPER_ARMS if RUN_PAPER_ARMS else [])
              + (ALPHA_ARMS if RUN_ALPHA_SWEEP else []))

PILOT_TAGS = []
if RUN_SWORDS_PILOT:
    for seed in SEEDS:
        for arm in PILOT_ARMS:
            PILOT_TAGS.append(train_one(arm, seed))
print('Pilot arms:', PILOT_ARMS)
print('Pilot tags:', PILOT_TAGS)


/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/run_clm_sequence_ncp.py --model_name_or_path /content/Llama-3.2-1B --tokenizer_name /content/Llama-3.2-1B --train_file /content/concept_aware_training/data/swords/concept_dev/context_loss_train.csv --validation_file /content/concept_aware_training/data/swords/concept_dev/context_loss_val.csv --objective none --ncp_alpha 0.0 --base_loss_weight 1.0 --required_coverage 0.99 --contrast_beta 0.0 --preprocessing_cache_dir /content/task14_scratch/cache_swords --optim adamw_torch --candidate_microbatch_size 8 --no-gradient_checkpointing --seed 42 --output_dir /content/task14_scratch/1B_swords_C0_s42 --num_train_epochs 2 --learning_rate 1e-5 --warmup_ratio 0.1 --block_size 128 --per_device_train_batch_size 1 --per_device_eval_batch_size 1 --gradient_accumulation_steps 16 --logging_steps 25 --save_strategy no --save_only_model --bf16 --overwrite_output_dir --do_train --do_eval --report_to none --forb

## 7. Paired comparisons and predeclared gate

In [ ]:
def paired_compare(benchmark, baseline_tag, treatment_tag, mode='left', prefix=None):
    # HyperLex is written per split (hyperlex_lexical__*, hyperlex_random__*), so the file prefix
    # and the --benchmark value are not the same string for it.
    prefix = prefix or ('bm_semlex' if benchmark == 'bm_semlex' else benchmark)
    output = RESULTS / f'paired_{prefix}__{treatment_tag}_vs_{baseline_tag}.json'
    run_command([sys.executable, SCRIPTS / 'compare_task14_results.py',
                 '--baseline_json', RESULTS / f'{prefix}__{baseline_tag}.json',
                 '--treatment_json', RESULTS / f'{prefix}__{treatment_tag}.json',
                 '--benchmark', benchmark, '--mode', mode, '--n_bootstrap', '5000',
                 '--results_json', output])
    return json.load(open(output))

def evaluate_seed_gate(seed):
    c0 = f'1B_swords_C0_s{seed}'
    mx = f'1B_swords_M_X_s{seed}'
    mi = f'1B_swords_M_I_s{seed}'
    mi_vs_c0 = paired_compare('swords', c0, mi)
    mi_vs_mx = paired_compare('swords', mx, mi)
    bm_mi_vs_c0 = paired_compare('bm_semlex', c0, mi)

    def metric(report, name): return report['metrics'].get(name, {})
    gap = metric(mi_vs_c0, 'gap_rat')
    alt = metric(mi_vs_c0, 'alternatives_nll')
    gold_vs_c0 = metric(mi_vs_c0, 'gold_nll')
    gold_vs_mx = metric(mi_vs_mx, 'gold_nll')
    single = metric(mi_vs_c0, 'acceptable_single_logp_mean')
    multi = metric(mi_vs_c0, 'acceptable_multi_logp_mean')
    bm = metric(bm_mi_vs_c0, 'correct_float')
    # compare_task14_results.py has been bootstrapping these two all along; nothing read them.
    p_at_1 = metric(mi_vs_c0, 'p_at_1')
    auroc = metric(mi_vs_c0, 'auroc')

    c0_intr = json.load(open(RESULTS / f'intrinsic__{c0}.json'))[0]
    mi_intr = json.load(open(RESULTS / f'intrinsic__{mi}.json'))[0]
    ntp_delta = mi_intr['ntp']['ntp_nll_mean'] - c0_intr['ntp']['ntp_nll_mean']
    gate = {
        'gap_ratio_positive': bool(gap and gap['ci95'][0] > 0),
        'alternatives_nll_improves': bool(alt and alt['ci95'][1] < 0),
        'observed_gold_nll_delta_le_0.20': bool(
            gold_vs_c0 and gold_vs_c0['treatment_minus_baseline'] <= 0.20
        ),
        'gold_retained_better_than_MX': bool(gold_vs_mx and gold_vs_mx['treatment_minus_baseline'] < 0),
        'general_ntp_delta_le_0.20': ntp_delta <= 0.20,
        'bm_wrong_sense_noninferior_5pp': bool(bm and bm['ci95'][0] > -0.05),
        'single_token_gain': bool(single and single['treatment_minus_baseline'] > 0),
        'multi_token_gain': bool(multi and multi['treatment_minus_baseline'] > 0),
        'ntp_delta': ntp_delta,
        # Recorded, NOT gated -- same treatment as ntp_delta above.  This gate was declared
        # before any result was read and it has already been evaluated; adding pass/fail
        # criteria to it now would let a post-hoc metric move a line that has already been
        # drawn, which is exactly what the predeclaration exists to prevent.  Report them,
        # and gate on them only in a NEW gate declared for the arm being carried forward.
        'p_at_1_delta': p_at_1.get('treatment_minus_baseline'),
        'p_at_1_ci95': p_at_1.get('ci95'),
        'auroc_delta': auroc.get('treatment_minus_baseline'),
        'auroc_ci95': auroc.get('ci95'),
    }
    OBSERVED_ONLY = {'ntp_delta', 'passes_all', 'p_at_1_delta', 'p_at_1_ci95',
                     'auroc_delta', 'auroc_ci95'}
    gate['passes_all'] = all(value for key, value in gate.items()
                              if key not in OBSERVED_ONLY)
    json.dump(gate, open(RESULTS / f'swords_pilot_gate_s{seed}.json', 'w'), indent=2)
    return gate

GATES = {seed: evaluate_seed_gate(seed) for seed in SEEDS} if RUN_SWORDS_PILOT else {}
GATE = GATES.get(PRIMARY_SEED, {})
ALL_SEEDS_PASS = bool(GATES) and all(gate.get('passes_all') for gate in GATES.values())
print(json.dumps({'per_seed': GATES, 'all_seeds_pass': ALL_SEEDS_PASS}, indent=2))


/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/compare_task14_results.py --baseline_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/swords__1B_swords_C0_s42.json --treatment_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/swords__1B_swords_M_I_s42.json --benchmark swords --mode left --n_bootstrap 5000 --results_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/paired_swords__1B_swords_M_I_s42_vs_1B_swords_C0_s42.json
/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/compare_task14_results.py --baseline_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/swords__1B_swords_M_X_s42.json --treatment_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/swords__1B_swords_M_I_s42.json --benchmark swords --mode left --n_bootstrap 5000 --results_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled

### 7b. Iyer et al. baselines on the external benchmarks

`M_I` is compared against each paper arm on SWORDS and bm-semlex. These are **reported, not
gates** — `evaluate_seed_gate` above keeps its predeclared criteria against `C0` and `M_X`, so
adding baselines cannot move the pass/fail line after the fact.

Arm ownership: `B0` untouched base, `A0`/`A1`/`P1` from Iyer et al., `C0`/`M_X`/`M_I` ours.

In [ ]:
def headline(prefix, tag, mode='left'):
    path = RESULTS / f'{prefix}__{tag}.json'
    if not path.exists():
        return {}
    record = json.load(open(path))[0]
    return record.get(mode, record)

# Every one of our arms against every paper arm, declared BEFORE any result is read.  Comparing
# only M_I would mean that if M_P or M_PC turns out to be the headline -- which Task 13's
# mechanism predicts -- its comparison against the paper would have to be added after the fact,
# which is precisely what the CORE_ARMS gate exists to prevent.
PAPER_COMPARISONS = {}
if RUN_SWORDS_PILOT and RUN_PAPER_ARMS:
    for seed in SEEDS:
        for ours in OUR_ARMS:
            treatment = f'1B_swords_{ours}_s{seed}'
            if not (RESULTS / f'swords__{treatment}.json').exists():
                continue
            for arm in PAPER_ARMS:
                baseline = f'1B_swords_{arm}_s{seed}'
                if not (RESULTS / f'swords__{baseline}.json').exists():
                    continue
                PAPER_COMPARISONS[f'{ours}_vs_{arm}_s{seed}'] = {
                    'swords': paired_compare('swords', baseline, treatment),
                    'bm_semlex': paired_compare('bm_semlex', baseline, treatment),
                }
print(f'paper comparisons: {len(PAPER_COMPARISONS)} '
      f'({len(OUR_ARMS)} ours x {len(PAPER_ARMS)} paper x {len(SEEDS)} seeds)')

# M_C vs M_I isolates the contrastive term on HUMAN-labelled negatives: same rows, same replay,
# same objective and alpha, only contrast_beta differs.  This is the clean counterpart to Task
# 13b's K1 vs A2, whose WordNet negatives include some valid substitutes.
OURS_COMPARISONS = {}
if RUN_SWORDS_PILOT:
    for seed in SEEDS:
        mi = f'1B_swords_M_I_s{seed}'
        for arm in ['M_P', 'M_C', 'M_PC']:
            tag = f'1B_swords_{arm}_s{seed}'
            if not (RESULTS / f'swords__{tag}.json').exists():
                continue
            OURS_COMPARISONS[f'{arm}_vs_M_I_s{seed}'] = {
                'swords': paired_compare('swords', mi, tag),
                'bm_semlex': paired_compare('bm_semlex', mi, tag),
                # Off-relation transfer: these arms are trained on SWORDS synonymy, so a flat
                # HyperLex result is the expected outcome and is reported as a retention check,
                # not as hypernym evidence.  Phase C is where HyperLex becomes the matched
                # benchmark, against arms trained on hyp/youtube_clean_gold.
                **{f'hyperlex_{split}': paired_compare(
                       'hyperlex', mi, tag, prefix=f'hyperlex_{split}')
                   for split in ['lexical', 'random']
                   if (RESULTS / f'hyperlex_{split}__{tag}.json').exists()},
            }
print('within-ours comparisons:', sorted(OURS_COMPARISONS))

def summary_row(arm, seed):
    tag = 'base_1B' if arm == 'B0' else f'1B_swords_{arm}_s{seed}'
    swords, bm = headline('swords', tag), headline('bm_semlex', tag)
    intrinsic_path = RESULTS / f'intrinsic__{tag}.json'
    intrinsic = json.load(open(intrinsic_path))[0] if intrinsic_path.exists() else {}
    return {
        'arm': arm,
        'owner': 'base' if arm == 'B0' else ARMS[arm]['owner'],
        'seed': '-' if seed is None else seed,
        'GAP ratio': swords.get('gap_rat'),
        # p@1 and AUROC are stored by eval_swords.py alongside GAP and were simply never read
        # here.  p@1 is the most reviewer-legible number on this benchmark -- is the model's
        # top-ranked substitute one a human accepted -- and AUROC is the separation of
        # acceptable from unacceptable substitutes, which is exactly the quantity the
        # contrastive arms claim to improve.  Reporting GAP alone hides both.
        'p@1': swords.get('p_at_1'),
        'AUROC': swords.get('auroc'),
        'spearman': swords.get('spearman'),
        'ALT NLL': swords.get('alternatives_nll'),
        'GOLD NLL': swords.get('gold_nll'),
        # Lower is better: the share of acceptable+rejected mass sitting on substitutes humans
        # marked unacceptable.  This is what the contrastive arms claim to reduce; AUROC only
        # shows those substitutes rank lower, not that their probability was pushed down.
        'REJ share': swords.get('rejected_mass_share'),
        'bm acc': bm.get('accuracy') if isinstance(bm, dict) else None,
        'hyp-lex dir': headline('hyperlex_lexical', tag).get('directionality_accuracy'),
        'hyp-rnd dir': headline('hyperlex_random', tag).get('directionality_accuracy'),
        'hyp-lex rho': headline('hyperlex_lexical', tag).get('spearman_gold'),
        'NTP NLL': (intrinsic.get('ntp') or {}).get('ntp_nll_mean'),
        # PPL is exp(NLL) and carries no extra information, but it is the unit the field reads,
        # so report it and keep NLL for the paired deltas.  Accuracy is NOT a transform of NLL:
        # it is independent signal, it is what Task 13b's master table reports, and leaving it
        # out of Task 14 alone makes the two notebooks non-comparable.
        'NTP PPL': (intrinsic.get('ntp') or {}).get('ntp_ppl'),
        'NTP acc': (intrinsic.get('ntp') or {}).get('ntp_accuracy'),
    }

# B0 is seed-independent, so it appears once rather than repeated per seed.
plan = ([('B0', None)] if RUN_PHASE_A else []) + (
    [(arm, seed) for seed in SEEDS for arm in PILOT_ARMS] if RUN_SWORDS_PILOT else [])
rows = [summary_row(arm, seed) for arm, seed in plan]

external_table = pd.DataFrame(rows)
external_table.to_csv(RESULTS / 'task14_external_summary.csv', index=False)
print(external_table.to_string(index=False, float_format=lambda value: f'{value:.4f}'))


/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/compare_task14_results.py --baseline_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/swords__1B_swords_A0_s42.json --treatment_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/swords__1B_swords_C0_s42.json --benchmark swords --mode left --n_bootstrap 5000 --results_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/paired_swords__1B_swords_C0_s42_vs_1B_swords_A0_s42.json
/usr/bin/python3 /content/concept_aware_training/transformers/examples/pytorch/language-modeling/compare_task14_results.py --baseline_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/bm_semlex__1B_swords_A0_s42.json --treatment_json /content/drive/MyDrive/concept_aware_outputs/task14_controlled/bm_semlex__1B_swords_C0_s42.json --benchmark bm_semlex --mode left --n_bootstrap 5000 --results_json /content/drive/MyDrive/concept_aware_outputs/task14_contr

### 7c. Paired deltas with confidence intervals

Sections 7 and 7b compute 63 paired bootstraps at 5,000 resamples each and print only their
names. Point estimates in the summary table cannot separate a real effect from seed noise;
these intervals can, and they are what the paper reports. `p@1` and `auroc` are included
because `compare_task14_results.py` has been bootstrapping them all along.

A claim is allowed only when every seed agrees in sign **and** every seed's CI excludes
zero -- the same two-part rule Task 13b's `claim_gate` uses, so the notebooks stay
comparable.


In [ ]:
# Read the paired reports that sections 7/7b already computed.  Nothing here retrains or
# re-evaluates: it reads the JSON that compare_task14_results.py wrote.
PAIRED_FIELDS = ['gap_rat', 'p_at_1', 'auroc', 'spearman',
                 'alternatives_nll', 'gold_nll', 'rejected_mass_share']
# Sign convention: for these, a NEGATIVE delta is the improvement (they are losses or an
# unwanted mass share).  Everything else in PAIRED_FIELDS improves upward.
LOWER_IS_BETTER = {'alternatives_nll', 'gold_nll', 'rejected_mass_share'}

def paired_frame(comparisons, benchmark='swords', fields=PAIRED_FIELDS):
    rows = []
    for name, reports in sorted(comparisons.items()):
        if benchmark not in reports:
            continue
        treatment, _, rest = name.partition('_vs_')
        baseline, _, seed = rest.rpartition('_s')
        for field in fields:
            stat = reports[benchmark]['metrics'].get(field)
            if not stat:
                continue
            low, high = stat['ci95']
            delta = stat['treatment_minus_baseline']
            rows.append({
                'treatment': treatment, 'baseline': baseline, 'seed': seed,
                'metric': field, 'n': stat['n'], 'delta': delta,
                'ci_low': low, 'ci_high': high,
                'excludes_0': bool(low > 0 or high < 0),
                'better': (delta < 0) if field in LOWER_IS_BETTER else (delta > 0),
            })
    return pd.DataFrame(rows)

def claim_gate(frame):
    """Allow a claim only when all seeds agree in direction and every CI excludes zero."""
    rows = []
    for (treatment, baseline, metric), group in frame.groupby(['treatment', 'baseline', 'metric']):
        consistent = bool(group.better.all() or (~group.better).all())
        rows.append({
            'treatment': treatment, 'baseline': baseline, 'metric': metric,
            'seeds': len(group), 'mean delta': group.delta.mean(),
            'direction': 'better' if group.better.all() else
                         ('worse' if (~group.better).all() else 'mixed'),
            'all_seeds_agree': consistent,
            'all_CIs_exclude_0': bool(group.excludes_0.all()),
            'claim_allowed': bool(consistent and group.better.all()
                                  and group.excludes_0.all()),
        })
    return pd.DataFrame(rows).sort_values(['metric', 'treatment', 'baseline'])

ALL_PAIRED = pd.concat(
    [frame for frame in (paired_frame(PAPER_COMPARISONS), paired_frame(OURS_COMPARISONS))
     if not frame.empty],
    ignore_index=True) if (PAPER_COMPARISONS or OURS_COMPARISONS) else pd.DataFrame()

if ALL_PAIRED.empty:
    print('No paired reports found -- run sections 7 and 7b first.')
else:
    ALL_PAIRED.to_csv(RESULTS / 'task14_paired_deltas.csv', index=False)
    CLAIMS = claim_gate(ALL_PAIRED)
    CLAIMS.to_csv(RESULTS / 'task14_claim_gate.csv', index=False)
    print(f'{len(ALL_PAIRED)} paired deltas over {ALL_PAIRED.n.iloc[0]} SWORDS targets, '
          f'{ALL_PAIRED.seed.nunique()} seeds\n')
    print('--- claims that survive all seeds (sign agreement + every CI excluding zero) ---')
    survivors = CLAIMS[CLAIMS.claim_allowed]
    print(survivors.to_string(index=False, float_format=lambda v: f'{v:+.4f}')
          if len(survivors) else '(none)')
    print('\n--- p@1 and AUROC, every contrast ---')
    focus = CLAIMS[CLAIMS.metric.isin(['p_at_1', 'auroc'])]
    print(focus.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))


## 8. Conditional Phase C — locked SWORDS test, hypernym transfer, and scale

Leave these flags disabled until all three 1B development seeds pass. The first conditional block
re-trains matched arms on all SWORDS dev and touches official test exactly once per trained model.
The second trains on boundary-safe `hyp/youtube_clean_gold` and reads HyperLex lexical/random as
relation-specific evaluation. bm-semlex remains a synonym guardrail, not hypernym evidence.

In [ ]:
FINAL_SWORDS_TAGS = []
if RUN_PHASE_C_SWORDS_TEST:
    assert RUN_EXTRA_SEEDS, 'Replicate the 1B dev result with seeds 7 and 123 before test access.'
    assert ALL_SEEDS_PASS, 'Every enabled 1B seed must pass before official SWORDS test access.'
    test_arms = arms_for_source(OUR_ARMS, 'swords_full')
    for seed in SEEDS:
        for arm in test_arms:
            FINAL_SWORDS_TAGS.append(
                train_one(arm, seed, source='swords_full', evaluation='test')
            )
print('Official-test tags:', FINAL_SWORDS_TAGS)

PHASE_C_TAGS = []
if RUN_PHASE_C_HYPERNYM:
    assert RUN_EXTRA_SEEDS and ALL_SEEDS_PASS, 'Replicate the 1B SWORDS result before expansion.'
    hyp_arms = arms_for_source(OUR_ARMS, 'hypernym')
    for seed in SEEDS:
        for arm in hyp_arms:
            PHASE_C_TAGS.append(train_one(arm, seed, source='hypernym'))
print('Phase C tags:', PHASE_C_TAGS)

SCALE_TAGS = []
if TRAIN_3B_AFTER_PASS:
    assert RUN_EXTRA_SEEDS and ALL_SEEDS_PASS, 'Replicate the 1B result before scaling.'
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    assert gpu_gb >= 35, f'3B concept training requires an A100-class GPU; found {gpu_gb:.1f} GiB.'
    for seed in SEEDS:
        for arm in OUR_ARMS:
            SCALE_TAGS.append(train_one(arm, seed, source='swords', model_key='3B'))
print('Conditional 3B tags:', SCALE_TAGS)


Official-test tags: []
Phase C tags: []
Conditional 3B tags: []


## 9. Final storage audit

In [ ]:
shutil.rmtree(SCRATCH, ignore_errors=True)
prohibited = ['*.safetensors', '*.bin', 'optimizer.pt', 'scheduler.pt',
              'rng_state.pth', 'scaler.pt']
leftovers = [path for pattern in prohibited
             for path in RESULTS.rglob(pattern)]
assert not leftovers, f'weight/optimizer artifacts reached Drive: {leftovers[:5]}'
print('Storage audit passed. Drive contains reports only:')
for path in sorted(RESULTS.iterdir()):
    print(f'{path.stat().st_size / 1024:9.1f} KB  {path.name}')


## Interpretation

- Treat the one-seed run as a development screen, not a paper result.
- The decisive concept metric is **gold-excluded acceptable-alternative NLL**, not inclusive NLL.
- `gap_rat` is the official-style rater-ratio GAP; `gap` uses raw TRUE counts. Oracle-k F1 is
  secondary and is labelled as such because k is taken from gold labels.
- Do not infer an annotation-quality effect by contrasting unmatched SWORDS and YouTube corpora.
- If the gate passes, enable seeds 7/123, lock the schedule, retrain on all SWORDS dev, evaluate
  official SWORDS test once, and only then consider 3B training on an A100.